# Fantasy Football Player Dropoff Prediction Pipeline

This notebook demonstrates how to use the `PlayerDropoffPipeline` class to predict which fantasy football players are at risk of declining performance in the upcoming season.

The pipeline analyzes historical player performance data to identify patterns that predict when players will experience significant fantasy point dropoffs (20%+ decline by default).

In [ ]:
import sys
sys.path.append('../src')

from player_dropoff_pipeline import PlayerDropoffPipeline
import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

## Basic Usage

Create a pipeline with recent seasons of data and run predictions:

In [ ]:
# Create pipeline with recent seasons (2018-2024)
seasons = list(range(2018, 2025))
pipeline = PlayerDropoffPipeline(seasons=seasons, dropoff_threshold=0.2)

# Run the pipeline to predict 2025 dropoffs
predictions, model_results = pipeline.run(predict_season=2025, save_csv=False)

## Model Performance

In [ ]:
print(f"Training AUC: {model_results['train_auc']:.3f}")
print(f"Test AUC: {model_results['test_auc']:.3f}")
print("\nTest Set Classification Report:")
print(model_results['test_report'])

## Feature Importance

Which factors are most predictive of fantasy dropoffs?

In [ ]:
# Display feature importance
import matplotlib.pyplot as plt

features = list(model_results['feature_importance'].keys())
importances = list(model_results['feature_importance'].values())

# Sort by importance
sorted_idx = sorted(range(len(importances)), key=lambda i: importances[i], reverse=True)
sorted_features = [features[i] for i in sorted_idx]
sorted_importances = [importances[i] for i in sorted_idx]

# Plot top 10 features
plt.figure(figsize=(10, 6))
plt.barh(sorted_features[:10], sorted_importances[:10])
plt.xlabel('Feature Importance')
plt.title('Top 10 Most Important Features for Predicting Fantasy Dropoffs')
plt.tight_layout()
plt.show()

# Print top features
print("Top 10 Feature Importances:")
for feature, importance in zip(sorted_features[:10], sorted_importances[:10]):
    print(f"{feature}: {importance:.3f}")

## Predictions Analysis

Let's examine the players predicted to have the highest dropoff risk:

In [ ]:
print("Top 20 Predicted Dropoffs for 2025:")
print(predictions[['player_name', 'position', 'recent_team', 'fantasy_points', 'dropoff_probability', 'risk_tier']].head(20))

## Risk Tier Analysis

Break down predictions by risk tier:

In [ ]:
# Risk tier breakdown
print("Risk Tier Distribution:")
print(predictions['risk_tier'].value_counts())
print()

# Position breakdown by risk
print("Risk by Position:")
risk_by_position = predictions.groupby(['position', 'risk_tier']).size().unstack(fill_value=0)
print(risk_by_position)
print()

# High risk players by position
high_risk = predictions[predictions['risk_tier'] == 'High Risk']
print(f"High Risk Players by Position:")
print(high_risk['position'].value_counts())

## Fantasy Point Analysis

Look at the fantasy point distribution for high-risk players:

In [ ]:
# Plot fantasy points vs dropoff probability
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.scatter(predictions['fantasy_points'], predictions['dropoff_probability'], alpha=0.6)
plt.xlabel('2024 Fantasy Points')
plt.ylabel('Dropoff Probability')
plt.title('Fantasy Points vs Dropoff Risk')

plt.subplot(1, 2, 2)
risk_tiers = ['Low Risk', 'Medium Risk', 'High Risk']
colors = ['green', 'yellow', 'red']
for tier, color in zip(risk_tiers, colors):
    tier_data = predictions[predictions['risk_tier'] == tier]
    plt.hist(tier_data['fantasy_points'], alpha=0.6, label=tier, color=color, bins=20)
plt.xlabel('2024 Fantasy Points')
plt.ylabel('Count')
plt.title('Fantasy Points Distribution by Risk Tier')
plt.legend()

plt.tight_layout()
plt.show()

## Position-Specific Analysis

Look at high-risk players by position:

In [ ]:
positions = ['QB', 'RB', 'WR', 'TE']
for pos in positions:
    pos_high_risk = high_risk[high_risk['position'] == pos].sort_values('dropoff_probability', ascending=False)
    if len(pos_high_risk) > 0:
        print(f"\nTop 5 High-Risk {pos}s:")
        print(pos_high_risk[['player_name', 'recent_team', 'fantasy_points', 'dropoff_probability']].head())

## Custom Analysis

You can customize the pipeline parameters:

In [ ]:
# Example: More conservative dropoff threshold (30% decline)
conservative_pipeline = PlayerDropoffPipeline(
    seasons=list(range(2020, 2025)),  # Use more recent data
    dropoff_threshold=0.3  # 30% decline threshold
)

conservative_predictions, conservative_results = conservative_pipeline.run(predict_season=2025, save_csv=False)

print(f"Conservative Model Performance:")
print(f"Test AUC: {conservative_results['test_auc']:.3f}")
print(f"\nTop 10 Conservative High-Risk Players:")
print(conservative_predictions[conservative_predictions['risk_tier'] == 'High Risk'].head(10)[['player_name', 'position', 'recent_team', 'dropoff_probability']])

## Save Results

Save predictions to CSV for further analysis:

In [ ]:
# Save to CSV
predictions.to_csv('dropoff_predictions_2025.csv', index=False)
print("Predictions saved to 'dropoff_predictions_2025.csv'")

# Save just high-risk players
high_risk_players = predictions[predictions['risk_tier'] == 'High Risk']
high_risk_players.to_csv('high_risk_players_2025.csv', index=False)
print(f"Saved {len(high_risk_players)} high-risk players to 'high_risk_players_2025.csv'")